In [ ]:
# =========================
# 1. IMPORTS
# =========================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models, layers

warnings.filterwarnings("ignore")

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# =========================
# 2. LOAD DATA
# =========================
address = "./Input/music genre.xlsx"

columns = [
            'filename',
            'tempo',
            'beats',
            'chroma_stft',
            'rmse',
            'spectral_centroid',
            'spectral_bandwidth',
            'rolloff',
            'zero_crossing_rate',
            'mfcc1',
            'mfcc2',
            'mfcc3',
            'mfcc4',
            'mfcc5',
            'mfcc6',
            'mfcc7',
            'mfcc8',
            'mfcc9',
            'mfcc10',
            'mfcc11',
            'mfcc12',
            'mfcc13',
            'mfcc14',
            'mfcc15',
            'mfcc16',
            'mfcc17',
            'mfcc18',
            'mfcc19',
            'mfcc20',
            'label'
            ]

df = pd.read_excel(address)
df.columns = columns

df = df.drop(columns=["filename"]).copy()
df = df.dropna().reset_index(drop=True)

# =========================
# 3. LABEL ENCODING
# =========================
class_names = sorted(df["label"].unique())
label_map = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {idx: name for name, idx in label_map.items()}

df["label"] = df["label"].map(label_map).astype("int32")

# =========================
# 4. SPLIT
# =========================
X = df.drop(columns=["label"])
y = df["label"]

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train_df.shape)
print("X_test :", X_test_df.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("Classes:", class_names)

# =========================
# 5. SCALING
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_df).astype(np.float32)
X_test = scaler.transform(X_test_df).astype(np.float32)

y_train = y_train.to_numpy(dtype=np.int32)
y_test = y_test.to_numpy(dtype=np.int32)

input_dim = X_train.shape[1]
num_classes = len(class_names)

print("Input dim :", input_dim)
print("Num classes:", num_classes)

# =========================
# 6. MODEL DEFINITION
# =========================
model = models.Sequential([
                            keras.Input(shape=(input_dim,)),
                            layers.Dense(1024, activation="relu"),
                            layers.Dropout(0.30),
                            layers.Dense(512, activation="relu"),
                            layers.Dropout(0.25),
                            layers.Dense(256, activation="relu"),
                            layers.Dense(num_classes, activation="softmax")
                        ])

optimizer = keras.optimizers.Adamax(learning_rate=0.0005,beta_1=0.9,beta_2=0.999,epsilon=1e-07,name="Adamax")
model.compile(optimizer=optimizer,loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.summary()


early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss",patience=100,restore_best_weights=True,verbose=1)

# =========================
# 7. CALLBACKS, TRAIN AND EVALUATION
# =========================
history = model.fit(X_train,y_train,validation_data=(X_test, y_test),epochs=500,batch_size=256,callbacks=[early_stopping],verbose=1,shuffle=True, steps_per_epoch=256)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nLoss = {test_loss:.4f}")
print(f"Accuracy = {test_accuracy:.4f}")

# =========================
# 8. PLOTS
# =========================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# =========================
# 9. PREDICTIONS + EXPORT
# =========================
pred_proba = model.predict(X_test, verbose=0)
pred_idx = np.argmax(pred_proba, axis=1)

results = X_test_df.copy()
results["label_true"] = [idx_to_label[i] for i in y_test]
results["label_predicted"] = [idx_to_label[i] for i in pred_idx]

results.to_csv("./Output/music_results_keras.csv", index=True, header=True)
results.head()